In [44]:
# ===============================
# import libraries
# ===============================
import numpy as np
import pandas as pd
from pathlib import Path
import chartgazing as cg

cg.set_size("4:5")

In [56]:
# ===============================
# import crad data
# ===============================

# set data directory
DATA_DIR = Path("data/sets")

# combine every saved set
pokemon_all = pd.concat(
    [pd.read_csv(path, dtype={"number": str, "product_id": str, "variant": str}, parse_dates=["date"]) for path in DATA_DIR.glob("*.csv")],
    ignore_index=True
)

# additional data on each set
sets_df = pd.read_csv("data/sets_info.csv", parse_dates=["release_date"])
pokemon_all = pokemon_all.merge(sets_df, on="set_slug", how="left")

In [46]:
# ===============================
# clean data
# ===============================
pokemon_all["variant"] = pokemon_all["variant"].fillna("Standard")
pokemon_all["price"] = pokemon_all["price"].replace(0, np.nan)
pokemon_all["date"] = pokemon_all["date"].dt.to_period("M").dt.to_timestamp()
pokemon_all["decade"] = (pokemon_all["release_date"].dt.year // 10) * 10


for col in ["grade", "set_slug", "set_name", "language", "name", "variant"]:
    pokemon_all[col] = pokemon_all[col].astype("category")

In [52]:
# ===============================
# Changes for all cards
# ===============================
GRADE = "Ungraded"
IMAGE_DIR = "images/4x5"
source = "Source: PriceCharting.com"

all_prices = pokemon_all.dropna(subset=["price"])

# TEMP LINE
# priced_cards = all_prices[all_prices["grade"] == GRADE]
priced_cards = all_prices

latest_date = priced_cards["date"].max()
latest_cards = priced_cards[priced_cards["date"] == latest_date]

def fmt(value):
    return f"${value:,.0f}" if value >= 1000 else f"${value:.2f}"

def short_set(set_name):
    return set_name.removeprefix("Pokemon ")

def one_per_card(cards):
    return(
        cards.assign(not_standard=cards["variant"] != "Standard")
        .sort_values(["not_standard", "price"])
        .drop_duplicates(["set_slug", "number", "name"])
    )

In [106]:
cards_2026 = all_prices[(all_prices["date"] >= "2026-01-01") & (all_prices["grade"] == GRADE)]

# cards_2026 = all_prices[all_prices["date"] >= "2026-01-01"]

year_start = cards_2026.loc[cards_2026["date"] == "2026-01-01", "product_id"].unique()

print(cards_2026.shape)

cards_2026 = cards_2026[cards_2026["product_id"].isin(year_start)]

cards_26 = cards_2026.groupby("date", as_index=False).agg(
    unique_cards = ("product_id", "nunique"),
    mean_price = ("price", "mean")
)

# cards_26["grade"] = cards_26["mean_price"].pct_change() * 100
# cards_26["absolute_chng"] = cards_26["mean_price"] - cards_26["mean_price"].iloc[0]

cards_26["PK_pct_chng"] = ((cards_26["mean_price"] - cards_26["mean_price"].iloc[0]) / cards_26["mean_price"].iloc[0]) * 100


cards_26

(394874, 13)


,date,unique_cards,mean_price,PK_pct_chng
0,2026-01-01,38445,17.343589,0.000000
1,2026-02-01,38444,17.705800,2.088445
2,2026-03-01,38444,18.400892,6.096221
3,2026-04-01,38441,19.262375,11.063371
4,2026-05-01,38442,20.528885,18.365843
5,2026-06-01,38442,21.498796,23.958170
6,2026-07-01,38442,22.286825,28.501804
7,2026-08-01,38443,23.442201,35.163497
8,2026-09-01,38442,24.478734,41.139956
9,2026-10-01,38441,25.859566,49.101586


In [110]:
# ===============================
# import S&P 500 data
# ===============================
sp_500 = pd.read_csv(f"data/S&P_500.csv")

sp_500["Date"] = pd.to_datetime(sp_500["Date"], format="%d/%m/%Y")
sp_500["Change"] = sp_500["Change %"].str.removesuffix("%") 
sp_500 = sp_500[["Date", "Price", "Change"]]

# sp_500 = sp_500.columns.str.lower()

sp_500["Change"] = sp_500["Change"].astype(float)
sp_500["Price"] = sp_500["Price"].str.replace(r"[^\d\.]", "",regex=True).astype(float)

sp_500 = sp_500[sp_500["Date"] >= "2026-01-01"]

sp_500 = sp_500.sort_values(by="Date", ascending=True)

sp_500["SP_pct_change"] = ((sp_500["Price"] - sp_500["Price"].iloc[0]) / sp_500["Price"].iloc[0]) * 100

sp_500.columns = sp_500.columns.str.lower()

sp_500


,date,price,change,sp_pct_change
10,2026-01-01,6939.03,1.37,0.000000
9,2026-02-01,6878.88,-0.87,-0.866836
8,2026-03-01,6528.52,-5.09,-5.915957
7,2026-04-01,7209.01,10.42,3.890746
6,2026-05-01,7580.06,5.15,9.238035
5,2026-06-01,7499.36,-1.06,8.075048
4,2026-07-01,7489.72,-0.13,7.936124
3,2026-08-01,7686.14,2.62,10.766779
2,2026-09-01,7651.54,-0.45,10.268150
1,2026-10-01,7580.07,-0.93,9.238179


In [114]:
returns = pd.merge(cards_26, sp_500, on='date')

returns = returns[["date", "PK_pct_chng", "sp_pct_change"]]

returns

,date,PK_pct_chng,sp_pct_change
0,2026-01-01,0.000000,0.000000
1,2026-02-01,2.088445,-0.866836
2,2026-03-01,6.096221,-5.915957
3,2026-04-01,11.063371,3.890746
4,2026-05-01,18.365843,9.238035
5,2026-06-01,23.958170,8.075048
6,2026-07-01,28.501804,7.936124
7,2026-08-01,35.163497,10.766779
8,2026-09-01,41.139956,10.268150
9,2026-10-01,49.101586,9.238179


In [89]:
all_prices.info()

purugly = all_prices[all_prices["name"].str.contains("purugly", case=False, na=False)]

purugly = purugly.sort_values("price", ascending=False)

purugly

<class 'pandas.DataFrame'>
Index: 7750954 entries, 15 to 13124129
Data columns (total 13 columns):
 #   Column        Dtype         
---  ------        -----         
 0   grade         category      
 1   date          datetime64[us]
 2   price         float64       
 3   set_slug      category      
 4   product_id    str           
 5   name          category      
 6   variant       category      
 7   number        str           
 8   set_name      category      
 9   language      category      
 10  release_date  datetime64[us]
 11  promo         bool          
 12  decade        int32         
dtypes: bool(1), category(6), datetime64[us](2), float64(1), int32(1), str(2)
memory usage: 532.2 MB


,grade,date,price,set_slug,product_id,name,variant,number,set_name,language,release_date,promo,decade
5379723,PSA 10,2026-09-01,267.31,pokemon-great-encounters,921310,purugly,Reverse Holo,50,Pokemon Great Encounters,English,2008-02-13,False,2000
5379724,PSA 10,2026-10-01,267.31,pokemon-great-encounters,921310,purugly,Reverse Holo,50,Pokemon Great Encounters,English,2008-02-13,False,2000
5379722,PSA 10,2026-08-01,250.17,pokemon-great-encounters,921310,purugly,Reverse Holo,50,Pokemon Great Encounters,English,2008-02-13,False,2000
5379721,PSA 10,2026-07-01,250.17,pokemon-great-encounters,921310,purugly,Reverse Holo,50,Pokemon Great Encounters,English,2008-02-13,False,2000
5379719,PSA 10,2026-05-01,200.34,pokemon-great-encounters,921310,purugly,Reverse Holo,50,Pokemon Great Encounters,English,2008-02-13,False,2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10861309,Ungraded,2026-09-01,0.12,pokemon-stellar-crown,7418602,purugly,Standard,117,Pokemon Stellar Crown,English,2024-09-13,False,2020
10861303,Ungraded,2026-03-01,0.10,pokemon-stellar-crown,7418602,purugly,Standard,117,Pokemon Stellar Crown,English,2024-09-13,False,2020
872964,Ungraded,2025-12-01,0.09,pokemon-astral-radiance,3489627,purugly,Standard,128,Pokemon Astral Radiance,English,2022-05-27,False,2020
1254282,Ungraded,2025-12-01,0.08,pokemon-battle-styles,1673296,purugly,Standard,116,Pokemon Battle Styles,English,2021-03-19,False,2020


In [51]:
GRADE_ORDER = ['Ungraded', 'Grade 7', 'Grade 8', 'Grade 9', 'Grade 9.5', 'PSA 10']